# 01 · Entendimento do negócio e dos dados
**CRISP-DM:** Entendimento do Negócio + Entendimento dos Dados (qualidade)

## 1. Entendimento do negócio
**Problema.** Na triagem de um incidente, o operador precisa saber se os recursos de API envolvidos estavam operando normalmente **antes do registro** e, se não estavam, **como, desde quando e de que lado** (provedor ou consumidor) está o desvio. Hoje essa informação depende de consultar várias ferramentas e de conhecer o comportamento típico de cada API, que muda com o horário, o dia da semana e os processamentos em lote.

**Objetivo de negócio.** Fornecer ao operador (ou ao agente que o apoia) uma informação explicável sobre o período anterior ao incidente, para direcionar a triagem com mais acerto. Não é monitoração em tempo real nem diagnóstico de causa raiz.

**Objetivo de mineração.** Para cada recurso de API: (1) caracterizar seus comportamentos normais a partir de Access Logs agregados; (2) classificar cada janela como **Normal** ou **Residual**; (3) descrever o residual (início, duração, atributos desviados, abrangência entre clientes, lado provável).

**Premissas.** A análise é sempre por recurso. Cada recurso pode ter vários comportamentos normais (dia útil, sábado, domingo/feriado, lote, manutenção recorrente). Há só dois estados: Normal e Residual.

**Questões de pesquisa.**
- **RQ1:** Access Logs agregados permitem caracterizar os regimes normais de cada recurso e separar o Residual do Normal melhor que abordagens sem contexto?
- **RQ2:** A caracterização do residual (início estimado e lado provável) é precisa o bastante para apoiar a triagem?
- **RQ3:** Quais atributos mais contribuem para identificar e caracterizar o residual?

**Critérios de sucesso** (definidos antes da modelagem):

| Tipo | Critério | Meta |
|---|---|---|
| Técnico | F1 por episódio versus o melhor baseline | maior, nas 10 sementes |
| Técnico | Janelas limpas marcadas como Residual | ≤ 2% |
| Técnico | Erro no início estimado | ≤ 1 janela (mediana) |
| Negócio | Acurácia do lado provável (episódios detectados) | ≥ 70% |
| Negócio | Consultas de controle (sem falha) com saída Normal | ≥ 95% |

**Riscos.** Não há histórico de incidentes (a avaliação usa falhas injetadas); o teste inclui um feriado (07/09); o treino tem só um mês.

In [1]:
%run ./00_funcoes.ipynb

00_funcoes carregado · raiz: /home/wsl/gh/mestrado/claude9


## 2. Entendimento dos dados: qualidade
Os dados são Access Logs já agregados por **minuto × recurso × status HTTP × cliente**. `TS_EVT` está em **UTC** e é convertido para o horário de Brasília.

In [2]:
bruto = carregar()
print(f"linhas: {len(bruto):,}")
print("período (Brasília):", bruto["ts"].min(), "→", bruto["ts"].max())
print("recursos:", bruto["recurso"].nunique(), "· clientes:", bruto["cliente"].nunique())
print("status:", sorted(bruto["status"].unique()))
print("nulos por coluna:", bruto.isna().sum().to_dict())
print("qtd ≤ 0:", int((bruto["qtd"] <= 0).sum()), "· tmp < 0:", int((bruto["tmp"] < 0).sum()))

linhas: 2,300,414
período (Brasília): 2026-07-31 21:00:00 → 2026-09-08 20:59:00
recursos: 12 · clientes: 1825
status: [np.int64(200), np.int64(201), np.int64(400), np.int64(401), np.int64(403), np.int64(412), np.int64(416), np.int64(422), np.int64(423), np.int64(429), np.int64(499), np.int64(500), np.int64(502), np.int64(503), np.int64(504)]
nulos por coluna: {'ts': 0, 'recurso': 0, 'status': 0, 'qtd': 0, 'tmp': 0, 'cliente': 0}
qtd ≤ 0: 0 · tmp < 0: 0


### Chaves repetidas
A mesma combinação minuto × recurso × status × cliente aparece em mais de uma linha (identificadores diferentes). Não são duplicatas exatas: parecem gravações parciais do mesmo minuto. Regra adotada: **consolidar** (somar o volume e usar a latência média ponderada).

In [3]:
chave = ["ts", "recurso", "status", "cliente"]
rep = bruto.duplicated(chave, keep=False)
qualidade = dict(
    linhas=len(bruto),
    linhas_em_chave_repetida=int(rep.sum()),
    linhas_excedentes=int(bruto.duplicated(chave).sum()),
    frac_volume_em_chave_repetida=float(bruto.loc[rep, "qtd"].sum() / bruto["qtd"].sum()),
    iguais_nas_metricas=int(bruto[rep].duplicated(chave + ["qtd", "tmp"]).sum()),
)
qualidade

{'linhas': 2300414,
 'linhas_em_chave_repetida': 25285,
 'linhas_excedentes': 17265,
 'frac_volume_em_chave_repetida': 0.009521663975675615,
 'iguais_nas_metricas': 13341}

### Cobertura temporal por recurso

In [4]:
cob = bruto.groupby("recurso").agg(inicio=("ts", "min"), fim=("ts", "max"), minutos=("ts", "nunique"),
                                   clientes=("cliente", "nunique"), volume=("qtd", "sum"))
cob["minutos_possiveis"] = ((cob["fim"] - cob["inicio"]) / pd.Timedelta("1min") + 1).astype(int)
cob["cobertura_min"] = cob["minutos"] / cob["minutos_possiveis"]
cob["part_volume"] = cob["volume"] / cob["volume"].sum()
cob.sort_values("volume", ascending=False)

,inicio,fim,minutos,clientes,volume,minutos_possiveis,cobertura_min,part_volume
recurso,,,,,,,,
18169917,2026-07-31 21:00:00,2026-09-08 20:59:00,56094,36,12939466,56160,0.998825,0.360549
18917906,2026-07-31 21:00:00,2026-09-03 07:11:00,47687,524,7195897,48132,0.990755,0.200508
18917920,2026-07-31 21:00:00,2026-09-03 07:11:00,47071,221,2057817,48132,0.977956,0.057340
18344302,2026-07-31 21:03:00,2026-09-08 20:59:00,44109,20,1762363,56157,0.785459,0.049107
18166630,2026-07-31 21:00:00,2026-09-08 20:59:00,52778,18,1723640,56160,0.939779,0.048028
18917921,2026-07-31 21:00:00,2026-09-03 07:11:00,47679,735,1701480,48132,0.990588,0.047410
18170306,2026-07-31 21:00:00,2026-09-08 20:58:00,46006,36,1675513,56159,0.819210,0.046687
18139260,2026-07-31 21:00:00,2026-09-08 20:59:00,56010,38,1641156,56160,0.997329,0.045730
18168196,2026-07-31 21:03:00,2026-09-08 20:59:00,49132,27,1485713,56157,0.874904,0.041398


Quatro recursos (`18917905`, `18917906`, `18917920`, `18917921`) têm o último registro em 03/09 às 07:11 (Brasília). No teste, eles só são avaliados até esse ponto.

### Volume e clientes por dia

In [5]:
dia = bruto.groupby(bruto["ts"].dt.date).agg(volume=("qtd", "sum"), clientes=("cliente", "nunique"),
                                            recursos=("recurso", "nunique"))
dia.index = pd.to_datetime(dia.index)
dia["dia_semana"] = dia.index.day_name()
dia

,volume,clientes,recursos,dia_semana
ts,,,,
2026-07-31,52824,162,12,Friday
2026-08-01,726178,264,12,Saturday
2026-08-02,831203,247,12,Sunday
2026-08-03,1113468,1074,12,Monday
2026-08-04,1155403,1060,12,Tuesday
2026-08-05,1099247,1054,12,Wednesday
2026-08-06,1040191,1095,12,Thursday
2026-08-07,1230929,1096,12,Friday
2026-08-08,835716,275,12,Saturday


### Distribuição de volume e latência (por linha)

In [6]:
desc = bruto[["qtd", "tmp"]].describe(percentiles=[0.5, 0.9, 0.99, 0.999]).T
extremos = dict(latencia_acima_60s=int((bruto["tmp"] > 60_000).sum()),
                latencia_acima_10s=int((bruto["tmp"] > 10_000).sum()),
                latencia_zero=int((bruto["tmp"] == 0).sum()))
display(desc)
extremos

,count,mean,std,min,50%,90%,99%,99.9%,max
qtd,2300414.0,15.600785,75.136563,1.0,3.0,22.0,248.0,1188.0,2165.0
tmp,2300414.0,77.476336,2858.080189,0.0,39.0,83.0,308.0,5009.0,1015962.0


{'latencia_acima_60s': 33, 'latencia_acima_10s': 271, 'latencia_zero': 1406}

### Proporção por família HTTP

In [7]:
fam = bruto.groupby(bruto["status"] // 100)["qtd"].sum()
fam_rec = bruto.assign(f=bruto["status"] // 100).pivot_table(index="recurso", columns="f", values="qtd",
                                                             aggfunc="sum", fill_value=0)
fam_rec = fam_rec.div(fam_rec.sum(axis=1), axis=0).round(3)
display((fam / fam.sum()).round(4))
fam_rec

status
2    0.7931
4    0.2040
5    0.0028
Name: qtd, dtype: float64

f,2,4,5
recurso,,,
14553746,0.983,0.009,0.008
17834257,0.105,0.884,0.011
18139260,0.909,0.090,0.001
18166630,0.993,0.006,0.000
18168196,0.992,0.006,0.001
18169917,0.993,0.005,0.002
18170306,0.970,0.029,0.002
18344302,0.894,0.105,0.000
18917905,0.882,0.118,0.000


## 3. Regras de validade adotadas
1. Conversão de `TS_EVT` de UTC para America/Sao_Paulo antes de qualquer corte temporal.
2. Consolidação das chaves repetidas (soma de volume, média de latência ponderada).
3. Linhas com volume ≤ 0 seriam descartadas (não há nenhuma).
4. Latências extremas são **mantidas**: são raras, podem ser eventos reais e o perfil usa estatísticas robustas (mediana/MAD).
5. Cada recurso é avaliado apenas até a sua última janela completa.

In [8]:
qualidade.update(extremos)
qualidade.update(periodo_inicio=bruto["ts"].min(), periodo_fim=bruto["ts"].max(),
                 recursos=int(bruto["recurso"].nunique()), clientes=int(bruto["cliente"].nunique()),
                 familias={int(k): float(v) for k, v in (fam / fam.sum()).items()},
                 recursos_encerrados=cob.loc[cob["fim"] < pd.Timestamp("2026-09-08"), "fim"].astype(str).to_dict())
salvar_json(qualidade, "01_qualidade.json")
cob.to_csv(RESULTADOS / "01_cobertura_recursos.csv")
dia.to_csv(RESULTADOS / "01_volume_diario.csv")
print("salvo em", RESULTADOS)

salvo em /home/wsl/gh/mestrado/claude9/resultados
